HW2 (Classical Text classification - Ch. 4): Construct at least two variant of
N-gram (justify your choice) and apply logistic regression on them and evaluate the results using proper metrics, justify your choice and analyze them.

In [1]:
import re
from typing import List
import zipfile
import os
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


Open the War and Piece book:

In [2]:
with open("warandpeace.txt", 'r') as f:
    wim_dataset = ' '.join(f.read().split())

Example of text:

In [3]:
wim_dataset[:300]

'Лев Николаевич Толстой Война и мир Первый вариант романа От автора Я пишу до сих пор только о князьях, графах, министрах, сенаторах и их детях и боюсь, что и вперед не будет других лиц в моей истории. Может быть, это нехорошо и не нравится публике; может быть, для нее интереснее и поучительнее истор'

In [4]:
def tokenize(text: str, pattern: str | None = None) -> List[str]:
    pattern = r"(?:\w+')\w+|(?:[A-Z]\.)+|\w+(?:-\w+)*|[\w+\.]"
    tokens = re.findall(pattern, text)
    return tokens

In [5]:
tokenize(wim_dataset)[440:]

['благодарю',
 'за',
 'него',
 'Бога',
 'но',
 'ежели',
 'счастье',
 'это',
 'не',
 'принадлежит',
 'всем',
 'то',
 'из',
 'этого',
 'я',
 'не',
 'вижу',
 'причины',
 'отрекаться',
 'от',
 'него',
 'и',
 'не',
 'пользоваться',
 'им',
 '.',
 'Я',
 'аристократ',
 'потому',
 'что',
 'не',
 'могу',
 'верить',
 'в',
 'высокий',
 'ум',
 'тонкий',
 'вкус',
 'и',
 'великую',
 'честность',
 'человека',
 'который',
 'ковыряет',
 'в',
 'носу',
 'пальцем',
 'и',
 'у',
 'которого',
 'душа',
 'с',
 'Богом',
 'беседует',
 '.',
 'Все',
 'это',
 'очень',
 'глупо',
 'может',
 'быть',
 'преступно',
 'дерзко',
 'но',
 'это',
 'так',
 '.',
 'И',
 'я',
 'вперед',
 'объявляю',
 'читателю',
 'какой',
 'я',
 'человек',
 'и',
 'чего',
 'он',
 'может',
 'ждать',
 'от',
 'меня',
 '.',
 'Еще',
 'время',
 'закрыть',
 'книгу',
 'и',
 'обличить',
 'меня',
 'как',
 'идиота',
 'ретрограда',
 'и',
 'Аскоченского',
 'которому',
 'я',
 'пользуясь',
 'этим',
 'случаем',
 'спешу',
 'заявить',
 'давно',
 'чувствуемое',
 'мно

Implement ngram algo

In [6]:
def add_bounds(tokens):
    start_token = "<S>"
    end_token = "<E>"
    tokens_new = [start_token]
    for i, token in enumerate(tokens):
        tokens_new.append(token)
        if token == ".":
            tokens_new.append(end_token)
            if i != len(tokens) - 1:
                tokens_new.append(start_token)
    if tokens[-1] == ".":
        tokens_new.append(end_token)
    return tokens_new

In [7]:
tokenized_text = tokenize(wim_dataset)
token_dataset = add_bounds(tokenized_text)

In [8]:
def ngram(tokens, n):
    counter = dict()
    for i in range(len(tokens) - n - 1):
        component = tuple(tokens[i: i + n])
        counter[component] = counter.get(component, dict())
        counter[component][tokens[i + n + 1]] = counter[component].get(tokens[i + n + 1], 0) + 1
    return counter


In [9]:
unigram = ngram(token_dataset[:5000], 0)
bigram = ngram(token_dataset[:5000], 1)
tgram = ngram(token_dataset[:5000], 2)

In [10]:
unigram_df = pd.DataFrame.from_dict(unigram, orient="index").fillna(0)
unigram_df

,Лев,Николаевич,Толстой,Война,и,мир,Первый,вариант,романа,От,...,пораженный,чем-то,необычайным,пожал,плечами,опустил,усаживалась,перед,освещала,неизменною
(),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
bigram_df = pd.DataFrame.from_dict(bigram, orient="index").fillna(0)
bigram_df

,Николаевич,быть,потому,и,.,так,не,аристократ,вижу,это,...,спины,собою,бала,кокетства,победительно,красоту,красоты,необычайным,пожал,опустил
<S>,1.0,1.0,6.0,3.0,1.0,3.0,6.0,4.0,2.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
и,0.0,1.0,0.0,5.0,14.0,1.0,6.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
не,0.0,1.0,0.0,5.0,3.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
быть,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
публике,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
победительно,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
умалить,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
пораженный,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
необычайным,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [12]:
tgram_df = pd.DataFrame.from_dict(tgram, orient="index").fillna(0)
tgram_df

,,Толстой,Война,и,мир,Первый,вариант,романа,От,автора,Я,...,красоту,желала,красоты,чем-то,необычайным,пожал,плечами,опустил,освещала,неизменною
<S>,Лев,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Лев,Николаевич,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Николаевич,Толстой,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
графах,министрах,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
и,их,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
чем-то,необычайным,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
необычайным,виконт,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
пожал,плечами,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
перед,ним,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Laplace

In [13]:
row_sum = bigram_df.sum(axis=1)
bigram_df_laplace = (bigram_df + 1).div(row_sum + bigram_df.shape[0], axis=0)
bigram_df_laplace

,Николаевич,быть,потому,и,.,так,не,аристократ,вижу,это,...,спины,собою,бала,кокетства,победительно,красоту,красоты,необычайным,пожал,опустил
<S>,0.000887,0.000887,0.003103,0.001773,0.000887,0.001773,0.003103,0.002216,0.001330,0.001773,...,0.000443,0.000443,0.000443,0.000443,0.000443,0.000443,0.000443,0.000443,0.000443,0.000443
и,0.000449,0.000898,0.000449,0.002695,0.006739,0.000898,0.003145,0.000449,0.000449,0.000449,...,0.000449,0.000449,0.000449,0.000449,0.000449,0.000449,0.000449,0.000449,0.000449,0.000449
не,0.000472,0.000943,0.000472,0.002830,0.001887,0.000472,0.002358,0.000472,0.000472,0.000472,...,0.000472,0.000472,0.000472,0.000472,0.000472,0.000472,0.000472,0.000472,0.000472,0.000472
быть,0.000494,0.000987,0.000494,0.000494,0.000987,0.000494,0.000494,0.000494,0.000494,0.000494,...,0.000494,0.000494,0.000494,0.000494,0.000494,0.000494,0.000494,0.000494,0.000494,0.000494
публике,0.000496,0.000992,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,...,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
победительно,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,...,0.000496,0.000496,0.000496,0.000496,0.000496,0.000992,0.000496,0.000496,0.000496,0.000496
умалить,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,...,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000992,0.000496,0.000496,0.000496
пораженный,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,...,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000992,0.000496,0.000496
необычайным,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,...,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000496,0.000992,0.000496


We can generate sentences based on most probable word with some distribution

In [14]:
import random

q = ngram(token_dataset, 1)
res = ""
s = "ну"
res += s
print("input token: ", s)
for i in range(100):
    if s == "<E>":
        break
    d = q[tuple([s])]
    tokens = list(d.keys())
    weights = list(d.values())
    s = random.choices(tokens, weights=weights, k=1)[0]
    res += " " + s
print("result:", res)

input token:  ну
result: ну . <S> до пор приходил нему вытянулся что так . <S> его направлены достижению сего усилия и же она глаза . <S> мой это чисто старательно в церковь . <S> мне вам хорошо <E>


But our main task is classification due to we use logistic regression

So, download another dataset

In [15]:
#!/bin/bash
!curl -L -o ~/Downloads/imdb-dataset-of-50k-movie-reviews.zip\
  https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 25.7M  100 25.7M    0     0  7953k      0  0:00:03  0:00:03 --:--:-- 11.9M


In [16]:
os.makedirs("dataset_imdb", exist_ok=True)


In [17]:
zip_path = os.path.expanduser('~/Downloads/imdb-dataset-of-50k-movie-reviews.zip')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('dataset_imdb')

In [18]:
imdb_data = pd.read_csv("dataset_imdb/IMDB Dataset.csv")

In [19]:
y = (imdb_data["sentiment"] == "positive").astype(int)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(imdb_data["review"], y, stratify=y, test_size=0.2, random_state=42)

In [21]:
vec1 = CountVectorizer(ngram_range=(1, 1), min_df=5)
X_train_1 = vec1.fit_transform(X_train)
X_test_1 = vec1.transform(X_test)

logreg = LogisticRegression(max_iter=1000).fit(X_train_1, y_train)

In [22]:
y_pred_1 = logreg.predict(X_test_1)

In [23]:
print(classification_report(y_test, y_pred_1))

              precision    recall  f1-score   support

           0       0.89      0.89      0.89      5000
           1       0.89      0.89      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [24]:
vec2 = CountVectorizer(ngram_range=(1, 2), min_df=5)
X_train_2 = vec2.fit_transform(X_train)
X_test_2 = vec2.transform(X_test)

logreg = LogisticRegression(max_iter=1000).fit(X_train_2, y_train)
y_pred_2 = logreg.predict(X_test_2)
print(classification_report(y_test, y_pred_2))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91      5000
           1       0.91      0.91      0.91      5000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000

